In [1]:
import pandas as pd 
import numpy as np 
from dotenv import load_dotenv
import os
import psutil

load_dotenv(override=True)

DATA_PATH = os.getenv('CSV_PATH')
STATIONS_PATH = os.getenv('STATIONS_PATH')

In [2]:
VALUE_COLS = [f'value{i}' for i in range(1, 32)]
USE_COLS = ['id', 'year', 'month', 'element'] + VALUE_COLS

DTYPES = {
    'id': 'string',
    'year': 'int16',
    'month': 'int8',
    'element': 'string',
}
for col in VALUE_COLS:
    DTYPES[col] = 'float32'

CHUNK_SIZE = 500_000

df = pd.read_csv(DATA_PATH, usecols=USE_COLS, dtype=DTYPES, na_values=[-9999], chunksize=CHUNK_SIZE)

chunk = next(df)

print(f"Shape of the chunk: {chunk.shape}")
print(f"Data types of the chunk:\n{chunk.dtypes}")
print(f"First 5 rows of the chunk:\n{chunk.head()}")

Shape of the chunk: (500000, 35)
Data types of the chunk:
id          string
year         int16
month         int8
element     string
value1     float32
value2     float32
value3     float32
value4     float32
value5     float32
value6     float32
value7     float32
value8     float32
value9     float32
value10    float32
value11    float32
value12    float32
value13    float32
value14    float32
value15    float32
value16    float32
value17    float32
value18    float32
value19    float32
value20    float32
value21    float32
value22    float32
value23    float32
value24    float32
value25    float32
value26    float32
value27    float32
value28    float32
value29    float32
value30    float32
value31    float32
dtype: object
First 5 rows of the chunk:
            id  year  month element  value1  value2  value3  value4  value5  \
0  ACW00011604  1949      1    TMAX   289.0   289.0   283.0   283.0   289.0   
1  ACW00011604  1949      2    TMAX   267.0   278.0   272.0   267.0   278.0   

## Percentagem de NULL

In [3]:
null_pct = (chunk.isnull().sum() / len(chunk) * 100).round(2)
null_df = null_pct.reset_index()
null_df.columns = ['coluna', 'pct_nulos (%)']

print("Percentagem de valores nulos por coluna:")
null_df

Percentagem de valores nulos por coluna:


,coluna,pct_nulos (%)
0,id,0.00
1,year,0.00
2,month,0.00
3,element,0.00
4,value1,9.96
5,value2,10.00
6,value3,9.95
7,value4,10.08
8,value5,9.94
9,value6,9.95


## Ano mais antigo

In [4]:
# Para processar todo o ficheiro por chunks e acumular min/max
year_agg = {}  # {id: {'min': ..., 'max': ...}}

reader = pd.read_csv(
    DATA_PATH,
    usecols=['id', 'year'],
    dtype={'id': 'string', 'year': 'int16'},
    chunksize=CHUNK_SIZE
)

for ck in reader:
    grp = ck.groupby('id')['year'].agg(['min', 'max'])
    for station_id, row in grp.iterrows():
        if station_id not in year_agg:
            year_agg[station_id] = {'min': row['min'], 'max': row['max']}
        else:
            year_agg[station_id]['min'] = min(year_agg[station_id]['min'], row['min'])
            year_agg[station_id]['max'] = max(year_agg[station_id]['max'], row['max'])

station_years = pd.DataFrame.from_dict(year_agg, orient='index')
station_years.index.name = 'id'
station_years.columns = ['ano_mais_antigo', 'ano_mais_recente']
station_years = station_years.reset_index()

print(f"Total de estações: {len(station_years)}")
station_years.head(10)

Total de estações: 40133


,id,ano_mais_antigo,ano_mais_recente
0,ACW00011604,1949,1949
1,ACW00011647,1961,1961
2,AE000041196,1944,2019
3,AEM00041194,1983,2019
4,AEM00041217,1983,2019
5,AEM00041218,1994,2019
6,AF000040930,1973,1992
7,AFM00040938,1973,2019
8,AFM00040948,1966,2019
9,AFM00040990,1973,2019


## Temperatura Media

In [6]:
# Calcular média ignorando NaN apenas sobre as colunas value1..value31
chunk['daily_avg_temp'] = chunk[VALUE_COLS].mean(axis=1)

print("Primeiras linhas com daily_avg_temp:")
chunk[['id', 'year', 'month', 'element', 'daily_avg_temp']].head(10)

Primeiras linhas com daily_avg_temp:


,id,year,month,element,daily_avg_temp
0,ACW00011604,1949,1,TMAX,274.612915
1,ACW00011604,1949,2,TMAX,271.142853
2,ACW00011604,1949,3,TMAX,277.935486
3,ACW00011604,1949,4,TMAX,287.166656
4,ACW00011604,1949,5,TMAX,291.354828
5,ACW00011604,1949,6,TMAX,294.833344
6,ACW00011604,1949,7,TMAX,298.709686
7,ACW00011647,1961,10,TMAX,272.000000
8,AE000041196,1944,3,TMAX,323.166656
9,AE000041196,1944,4,TMAX,321.466675


## Grupos 


In [7]:
temp_by_station_year = (
    chunk
    .groupby(['id', 'year'])['daily_avg_temp']
    .mean()
    .round(2)
    .reset_index()
)

print("Temperatura média anual por estação:")
temp_by_station_year.head(15)

Temperatura média anual por estação:


,id,year,daily_avg_temp
0,ACW00011604,1949,285.109985
1,ACW00011647,1961,272.000000
2,AE000041196,1944,348.869995
3,AE000041196,1945,318.230011
4,AE000041196,1955,317.920013
5,AE000041196,1956,318.109985
6,AE000041196,1957,311.390015
7,AE000041196,1958,317.899994
8,AE000041196,1959,309.899994
9,AE000041196,1960,316.929993


## Filtrar as 5 estações

In [8]:
# Ler o ficheiro de estações (largura fixa)
# Formato: ID (0-11), LAT (12-20), LON (21-30), ELEV (31-37), STATE (38-40),
#          NAME (41-71), GSFLAG (72-75), HCNFLAG (76-79), WMOID (80-85)
stations_path = STATIONS_PATH
if not os.path.exists(stations_path):
    alt_path = os.path.splitext(stations_path)[0] + '.txt'
    if os.path.exists(alt_path):
        stations_path = alt_path
    else:
        raise FileNotFoundError(
            f"Stations file not found: {stations_path!r}. "
            f"Verify STATIONS_PATH or put the file in the data folder."
        )

stations = pd.read_fwf(
    stations_path,
    colspecs=[(0, 11), (12, 20), (21, 30), (31, 37), (38, 40), (41, 71)],
    names=['id', 'lat', 'lon', 'elev', 'state', 'name'],
    dtype={'id': 'string', 'name': 'string'}
)
stations['name'] = stations['name'].str.strip()

# IDs das 5 estações portuguesas (verificar no ficheiro de estações)
PT_NAMES = ['HORTA', 'FUNCHAL', 'LISBOA', 'CASTELO BRANCO', 'FARO']

pt_stations = stations[stations['name'].str.upper().isin(PT_NAMES)]
print("Estações portuguesas encontradas:")
print(pt_stations[['id', 'name']])

PT_IDS = pt_stations['id'].tolist()

Estações portuguesas encontradas:
                id            name
25404  CA002100515            FARO
25405  CA002100516            FARO
47045  PO000008522         FUNCHAL
47065  POM00008554            FARO
47067  POM00008570  CASTELO BRANCO
50472  SWE00138286            FARO


In [17]:
# Ler o dataset completo filtrando apenas essas 5 estações
pt_chunks = []

reader3 = pd.read_csv(
    DATA_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    na_values=[-9999],
    chunksize=CHUNK_SIZE
)

for ck in reader3:
    filtered = ck[ck['id'].isin(PT_IDS)]
    if not filtered.empty:
        pt_chunks.append(filtered)

df_pt = pd.concat(pt_chunks, ignore_index=True)
print(f"Registos das estações portuguesas: {df_pt.shape}")
df_pt.head()

Registos das estações portuguesas: (1633, 35)


,id,year,month,element,value1,value2,value3,value4,value5,value6,...,value22,value23,value24,value25,value26,value27,value28,value29,value30,value31
0,CA002100515,1966,4,TMAX,11.0,50.0,67.0,61.0,56.0,61.0,...,-17.0,-22.0,6.0,-6.0,-17.0,28.0,44.0,28.0,50.0,NaN
1,CA002100515,1966,5,TMAX,67.0,67.0,56.0,61.0,83.0,156.0,...,50.0,50.0,72.0,72.0,72.0,56.0,67.0,89.0,122.0,139.0
2,CA002100515,1966,6,TMAX,128.0,167.0,172.0,139.0,156.0,183.0,...,183.0,178.0,222.0,195.0,178.0,100.0,156.0,156.0,172.0,NaN
3,CA002100515,1966,7,TMAX,206.0,178.0,150.0,167.0,228.0,222.0,...,NaN,NaN,217.0,200.0,189.0,167.0,117.0,122.0,117.0,161.0
4,CA002100515,1966,8,TMAX,128.0,122.0,167.0,172.0,217.0,234.0,...,133.0,150.0,139.0,133.0,128.0,161.0,167.0,139.0,122.0,117.0


## Substituir IDs

In [18]:
# Criar mapeamento id -> nome
id_to_name = dict(zip(pt_stations['id'], pt_stations['name']))

df_pt['id'] = df_pt['id'].map(id_to_name)
df_pt = df_pt.rename(columns={'id': 'station_name'})

print("Estações únicas no dataframe:")
print(df_pt['station_name'].unique())
print()
df_pt[['station_name', 'year', 'month', 'element']].head(10)

Estações únicas no dataframe:
<StringArray>
['FARO', 'FUNCHAL', 'CASTELO BRANCO']
Length: 3, dtype: str



,station_name,year,month,element
0,FARO,1966,4,TMAX
1,FARO,1966,5,TMAX
2,FARO,1966,6,TMAX
3,FARO,1966,7,TMAX
4,FARO,1966,8,TMAX
5,FARO,1966,9,TMAX
6,FARO,1966,10,TMAX
7,FARO,1966,11,TMAX
8,FARO,1966,12,TMAX
9,FARO,1967,1,TMAX


In [19]:
print(df_pt.columns.tolist())
print(df_pt.head(2))

['station_name', 'year', 'month', 'element', 'value1', 'value2', 'value3', 'value4', 'value5', 'value6', 'value7', 'value8', 'value9', 'value10', 'value11', 'value12', 'value13', 'value14', 'value15', 'value16', 'value17', 'value18', 'value19', 'value20', 'value21', 'value22', 'value23', 'value24', 'value25', 'value26', 'value27', 'value28', 'value29', 'value30', 'value31']
  station_name  year  month element  value1  value2  value3  value4  value5  \
0         FARO  1966      4    TMAX    11.0    50.0    67.0    61.0    56.0   
1         FARO  1966      5    TMAX    67.0    67.0    56.0    61.0    83.0   

   value6  ...  value22  value23  value24  value25  value26  value27  value28  \
0    61.0  ...    -17.0    -22.0      6.0     -6.0    -17.0     28.0     44.0   
1   156.0  ...     50.0     50.0     72.0     72.0     72.0     56.0     67.0   

   value29  value30  value31  
0     28.0     50.0      NaN  
1     89.0    122.0    139.0  

[2 rows x 35 columns]


## Ex2

### Q1 — Escolha e caracterização do dataset

In [20]:
taxi_csv = os.getenv('TAXI_PATH')

# Tamanho em disco
print(f"Tamanho do ficheiro: {os.path.getsize(taxi_csv) / 1024**3:.2f} GB")

# Contar linhas sem carregar tudo para memória
with open(taxi_csv, 'r') as f:
    n_lines = sum(1 for _ in f) - 1  # -1 para o cabeçalho

print(f"Total de linhas: {n_lines:,}")

Tamanho do ficheiro: 3.29 GB
Total de linhas: 22,288,907


In [21]:
# O dataset cabe em memória?
ram = psutil.virtual_memory()
tamanho_gb = os.path.getsize(taxi_csv) / 1024**3
n_linhas = 22_000_000

print("=" * 55)
print(f"  Tamanho em disco:      {tamanho_gb:.2f} GB")
print(f"  Total de linhas:       {n_linhas:,}")
print(f"  RAM total:             {ram.total / 1024**3:.1f} GB")
print(f"  RAM disponível:        {ram.available / 1024**3:.1f} GB")
print("=" * 55)

  Tamanho em disco:      3.29 GB
  Total de linhas:       22,000,000
  RAM total:             31.7 GB
  RAM disponível:        15.3 GB


### Q2 — Estratégia de leitura dos dados

**Caracterização do dataset:**
- Tamanho em disco: 3.29 GB
- Total de linhas: ~22 milhões
- Resultado da junção de 4 ficheiros mensais (Jan/2015, Jan/2016, Fev/2016, Mar/2016)

**O dataset cabe em memória?**  
Com 32 GB de RAM disponíveis, o ficheiro de 3.29 GB em disco cabe em memória. 
No entanto, ao carregar para pandas sem otimização, um CSV de 3.29 GB pode 
expandir para 8–10 GB em RAM (pandas usa float64 e object por defeito). 
Por isso, a estratégia adotada combina:
- `usecols` — carregar apenas as colunas necessárias para a análise
- `dtype` — forçar tipos mais pequenos (float32, int16, category)
- `chunksize` — processar em chunks de 500k linhas para operações pesadas

**Que colunas são realmente necessárias?**  
Das 19 colunas disponíveis, selecionamos 10 relevantes para a análise:
tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, trip_distance,
payment_type, fare_amount, tip_amount, total_amount, pickup_longitude, pickup_latitude

**Que tipos de dados podem ser otimizados?**  
- Valores monetários e distância: float64 → float32 (50% menos memória)
- passenger_count: int64 → int8 (valores 0–6)
- payment_type: int64 → category (só 5 valores distintos)

**É necessário processar por partes?**  
Para operações simples (leitura, filtros, criação de variáveis) não é necessário. 
Para agregações globais sobre as 22M linhas usamos chunksize para evitar picos de memória.

**Existem ficheiros que podem ser tratados separadamente?**  
Sim — os 4 ficheiros originais (por mês) foram tratados individualmente antes 
do merge, o que permitiu normalizar o nome da coluna RateCodeID/RatecodeID 
que diferia entre o ficheiro de 2015 e os de 2016.

### Q3 — Leitura de uma amostra ou chunk

O ficheiro `yellow_taxi_merged.csv` possui um tamanho de 3,29GB com cerca de 22 milhões de linhas. Por isso, é utilizado o `chunksize=500_000` para que não seja preciso carregar todo o ficheiro na memória simultaneamente. O chunk inicial serve para validar a configuração das colunas, tipos de dados e valores nulos.

In [22]:
TAXI_CHUNK_SIZE = 500_000

TAXI_READER = pd.read_csv(taxi_csv, chunksize=TAXI_CHUNK_SIZE)
taxi_chunk = next(TAXI_READER)

print(f"\nPrimeiras linhas do chunk:")
print(f"Número de linhas e colunas: {taxi_chunk.shape}")
print(f"\nNomes das colunas:\n{taxi_chunk.columns.tolist()}")
taxi_chunk.head()



Primeiras linhas do chunk:
Número de linhas e colunas: (500000, 19)

Nomes das colunas:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'RatecodeID', 'store_and_fwd_flag', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount']


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
0,2,2016-01-01 00:00:00,2016-01-01 00:00:00,2,1.10,-73.990372,40.734695,1,N,-73.981842,40.732407,2,7.5,0.5,0.5,0.0,0.0,0.3,8.8
1,2,2016-01-01 00:00:00,2016-01-01 00:00:00,5,4.90,-73.980782,40.729912,1,N,-73.944473,40.716679,1,18.0,0.5,0.5,0.0,0.0,0.3,19.3
2,2,2016-01-01 00:00:00,2016-01-01 00:00:00,1,10.54,-73.984550,40.679565,1,N,-73.950272,40.788925,1,33.0,0.5,0.5,0.0,0.0,0.3,34.3
3,2,2016-01-01 00:00:00,2016-01-01 00:00:00,1,4.75,-73.993469,40.718990,1,N,-73.962242,40.657333,2,16.5,0.0,0.5,0.0,0.0,0.3,17.3
4,2,2016-01-01 00:00:00,2016-01-01 00:00:00,3,1.76,-73.960625,40.781330,1,N,-73.977264,40.758514,2,8.0,0.0,0.5,0.0,0.0,0.3,8.8


**Descrição das variáveis principais:**
- `tpep_pickup_datetime` / `tpep_dropoff_datetime` - horário e data de incio e fim do trajeto
- `passenger_count` - quantidade de passageiros
- `trip_distance` - distância percorrida(milhas)
- `payment_type` - forma de pagamento (1 - cartão, 2 - dinheiro, 3 - sem cobrança, 4 - disputa)
- `fare_amount` - tarifa básica do trajeto
- `tip_amount` - gorjeta paga
- `total_amount` - valor total cobrado ao passageiro

#### Q4 - Otimização dos tipos de dados

##### **1. Análise do tipo de dados e memória antes da otimização**
Antes da otimização, analisámos os tipos de dados definidos pelo pandas e a memória usada pelo chunk, para servir de comparação

In [23]:
print("Tipos de dados:")
print(taxi_chunk.dtypes)
print(f"\nMemória total: {taxi_chunk.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Tipos de dados:
VendorID                   int64
tpep_pickup_datetime         str
tpep_dropoff_datetime        str
passenger_count            int64
trip_distance            float64
pickup_longitude         float64
pickup_latitude          float64
RatecodeID                 int64
store_and_fwd_flag           str
dropoff_longitude        float64
dropoff_latitude         float64
payment_type               int64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
improvement_surcharge    float64
total_amount             float64
dtype: object

Memória total: 149.73 MB


##### **2. Aplicar otimizações e comparar memória**
De acordo com o resultado  da análise, conseguimos detetar as colunas que requerem mudança do tipo de dado. Transformamos as colunas com datas do tipo texto para `datetime64`, as colunas com inteiros com um número pequeno de categorias para `int8`, a coluna com somente dois valores para `category` e todas as colunas `float` para `float32`.

In [24]:
taxi_otimizado = taxi_chunk.copy()

taxi_otimizado['tpep_pickup_datetime']  = pd.to_datetime(taxi_otimizado['tpep_pickup_datetime'])
taxi_otimizado['tpep_dropoff_datetime'] = pd.to_datetime(taxi_otimizado['tpep_dropoff_datetime'])

taxi_otimizado['VendorID']           = taxi_otimizado['VendorID'].astype('int8')
taxi_otimizado['passenger_count']    = taxi_otimizado['passenger_count'].astype('int8')
taxi_otimizado['RatecodeID']         = taxi_otimizado['RatecodeID'].astype('int8')
taxi_otimizado['payment_type']       = taxi_otimizado['payment_type'].astype('int8')
taxi_otimizado['store_and_fwd_flag'] = taxi_otimizado['store_and_fwd_flag'].astype('category')

colunas_float = ['trip_distance', 'pickup_longitude', 'pickup_latitude',
                 'dropoff_longitude', 'dropoff_latitude', 'fare_amount',
                 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
                 'improvement_surcharge', 'total_amount']

taxi_otimizado[colunas_float] = taxi_otimizado[colunas_float].astype('float32')

mem_antes = taxi_chunk.memory_usage(deep=True).sum() / 1024**2
mem_depois = taxi_otimizado.memory_usage(deep=True).sum() / 1024**2

print(f"Memória antes:  {mem_antes:.2f} MB")
print(f"Memória depois: {mem_depois:.2f} MB")
print(f"Redução:        {(1 - mem_depois / mem_antes) * 100:.1f}%")
print("\nNovos tipos:")
print(taxi_otimizado.dtypes)

Memória antes:  149.73 MB
Memória depois: 32.90 MB
Redução:        78.0%

Novos tipos:
VendorID                           int8
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                    int8
trip_distance                   float32
pickup_longitude                float32
pickup_latitude                 float32
RatecodeID                         int8
store_and_fwd_flag             category
dropoff_longitude               float32
dropoff_latitude                float32
payment_type                       int8
fare_amount                     float32
extra                           float32
mta_tax                         float32
tip_amount                      float32
tolls_amount                    float32
improvement_surcharge           float32
total_amount                    float32
dtype: object


**Resumo das otimizações realizadas:**

| Coluna| Antes | Depois | Porquê? |
|---|---|---|---|
|`tpep_pickup_datetime`, `tpep_dropoff_datetime` | `str` | `datetime64` | As datas guardadas como texto. Ao convertê-las para formato de data, o uso de memória fica mais eficiente. |
| `VendorID`, `passenger_count`, `RatecodeID`, `payment_type` | `int64` | `int8` | Estas colunas têm valores pequenos, por isso não precisam de ocupar tanto espaço de memória.
| `store_and_fwd_flag` | `str` | `category` | Esta coluna só tem dois valores possiveis, `"Y"` e `"N"`, por isso o tipo `category` é mais adequado.
| 12 colunas numéricas, como `fare_amount`, coordenadas, etc..| `float` | `float32` | A precisão continua a ser suficiente e o consumo de memória passa a ser menor.

Depois destas alterações, a memória usada pelo chunk passou de **149.73 MB** para **32.90 MB**, o que representa uma redução de cerca de **78%**

##### **3. Aplicar ao dataset completo**
Depois de validar as otimizações no chunk, implementamos as mesmas para a leitura do dataset completo, através do parâmetro `dtype` do `read_csv`. Assim, o pandas já carrega os dados com os tipos corretos desde o inicio, evitando um pico de memória caso a conversão fosse feita só depois da leitura.

In [25]:
TAXI_DTYPES = {
    'VendorID': 'int8',
    'passenger_count': 'int8',
    'RatecodeID': 'int8',
    'payment_type': 'int8',
    'store_and_fwd_flag': 'category',
    'trip_distance': 'float32',
    'pickup_longitude': 'float32',
    'pickup_latitude': 'float32',
    'dropoff_longitude': 'float32',
    'dropoff_latitude': 'float32',
    'fare_amount': 'float32',
    'extra': 'float32',
    'mta_tax': 'float32',
    'tip_amount': 'float32',
    'tolls_amount': 'float32',
    'improvement_surcharge': 'float32',
    'total_amount': 'float32',
}

df_taxi = pd.read_csv(
    taxi_csv,
    dtype=TAXI_DTYPES,
    parse_dates=['tpep_pickup_datetime', 'tpep_dropoff_datetime']
)

print(f"Shape: {df_taxi.shape}")
print(f"Memória total: {df_taxi.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Shape: (22288907, 19)
Memória total: 1466.69 MB


### Q5 - Valores em falta

#####  **1. Calcular a percentagem de NaN por coluna**
Iniciamos pela identificação da presença de dados nulos (NaN) na tabela completa. Isto é importante para conseguirmos perceber a dimensão inicial do problema da falta de dados.

In [26]:
null_pct = (df_taxi.isnull().sum() / len(df_taxi) * 100).round(2)
null_df = null_pct.reset_index()
null_df.columns = ['coluna', 'pct_nulos (%)']
print("Percentagem de valores nulos por coluna:")
null_df

Percentagem de valores nulos por coluna:


,coluna,pct_nulos (%)
0,VendorID,0.0
1,tpep_pickup_datetime,0.0
2,tpep_dropoff_datetime,0.0
3,passenger_count,0.0
4,trip_distance,0.0
5,pickup_longitude,0.0
6,pickup_latitude,0.0
7,RatecodeID,0.0
8,store_and_fwd_flag,0.0
9,dropoff_longitude,0.0


##### **2.  identificação de códigos especiais de missing values, se existirem.**
Embora não haja NaNs explicitos no conjunto de dados, é possível que existam valores usados como códigos de dados em falta, ou seja, valores numéricos que são válidos, mas que não têm significado real.

Por isso, analisámos alguns casos comuns neste tipo de dataset : viagens sem passageiro, distância igual a zero, coordenadas com valor zero e tarifas negativas ou nulas.

In [27]:
n = len(df_taxi)

print(f"passenger_count = 0:        {(df_taxi['passenger_count'] == 0).sum():,} ({(df_taxi['passenger_count'] == 0).sum()/n*100:.2f}%)")
print(f"trip_distance = 0:          {(df_taxi['trip_distance'] == 0).sum():,} ({(df_taxi['trip_distance'] == 0).sum()/n*100:.2f}%)")
print(f"pickup_longitude = 0:       {(df_taxi['pickup_longitude'] == 0).sum():,} ({(df_taxi['pickup_longitude'] == 0).sum()/n*100:.2f}%)")
print(f"pickup_latitude = 0:        {(df_taxi['pickup_latitude'] == 0).sum():,} ({(df_taxi['pickup_latitude'] == 0).sum()/n*100:.2f}%)")
print(f"fare_amount <= 0:           {(df_taxi['fare_amount'] <= 0).sum():,} ({(df_taxi['fare_amount'] <= 0).sum()/n*100:.2f}%)")
print(f"total_amount <= 0:          {(df_taxi['total_amount'] <= 0).sum():,} ({(df_taxi['total_amount'] <= 0).sum()/n*100:.2f}%)")

passenger_count = 0:        1,041 (0.00%)
trip_distance = 0:          131,749 (0.59%)
pickup_longitude = 0:       347,046 (1.56%)
pickup_latitude = 0:        347,046 (1.56%)
fare_amount <= 0:           15,016 (0.07%)
total_amount <= 0:          10,387 (0.05%)


##### **3. decisão sobre o tratamento dos valores em falta**
Com base nos valores encontrados, testámos duas formas de tratamento : remover esses registos ou substituir os valores pela mediana. O objetivo foi perceber qual das opções fazia mais sentido para este dataset.

In [ ]:
df_taxi_clean = df_taxi[
    (df_taxi['passenger_count'] > 0) &
    (df_taxi['trip_distance'] > 0) &
    (df_taxi['pickup_longitude'] != 0) &
    (df_taxi['pickup_latitude'] != 0) &
    (df_taxi['fare_amount'] > 0) &
    (df_taxi['total_amount'] > 0)
].copy()

print(f"Linhas antes:     {len(df_taxi):,}")
print(f"Linhas depois:    {len(df_taxi_clean):,}")
print(f"Linhas removidas: {len(df_taxi) - len(df_taxi_clean):,} ({(1 - len(df_taxi_clean)/len(df_taxi))*100:.2f}%)")

Linhas antes:     22,288,907
Linhas depois:    21,822,582
Linhas removidas: 466,325 (2.09%)


Foram testadas duas formas de tratar os valores problemáticos encontrados:
| | Remoção | Substituição pela mediana |
| --- | --- | --- |
| Linhas mantidas | 21,822,582 | 22,288,907 |
| Perda de dados | 2.09% | 0% |
| Coordenadas inválidas | Removidas | Substituídas pela mediana de NYC|
| Tarifas < 0 | Removidas | Substituídas pela mediana |

**Decisão : Remoção**
A opção escolhida foi remover as linhas com valores inválidos. As coordenadas `(0,0)` não fazem sentido num dataset de taxis de Nova Iorque e substituí-las pela mediana poderia criar localizações artificias. O registo mesmo acontece com viagens de distância zero ou com tarifas negativas, que parecem ser erros de registo e não acrescentam valor à análise.

Como a remoção afeta apenas **2.09%** de mais de **22 milhões de linhas**, a perda de dados é reduzida e não compromete a representatividadr do datest. Assim, o dataframe final, `df_taxi_clean`, fica apenas com registos considerados válidos.